# Inges notebook for Unknown stealer DIVD-2026-0002
This notebook takes *.jsonl files and reads them into a breach specific database. Before you run this make sure you update the constants below.

Record format:
```json
{"_index": "passwords", "_id": "FR__J5sB3iRUe_QmbOn4", "_score": null, "_source": {"report_id": 35, "user_id": 1, "soft_type": 1, "soft_name": "Google_Chrome_Profile 5", "url": "https://agent.xxx.io/xx/", "user": "xx@xxx.loc", "password": "xxxx"}, "sort": [0]}
{"_index": "passwords", "_id": "Fh__J5sB3iRUe_QmbOn4", "_score": null, "_source": {"report_id": 35, "user_id": 1, "soft_type": 1, "soft_name": "Google_Chrome_Profile 5", "url": "https://mail.xxx.xxx/xxx/", "user": "xxx@xxx.fr", "password": "ugoq1T2iCwHF"}, "sort": [1]}
```

# Import helper functions

In [ ]:
%run ../0.shared_notebooks/0_helper_functions.ipynb

# Set constants

In [ ]:
# change these
CASE="DIVD-2026-00002"
SUB="9202-202602191046"
PASSWORD_PREMASKED=False      # Set to true is the password in the normalized data is allready premasked
# Keep the same
IN_DIR=f"../IN/2026-00002-slurp/{SUB}/passwords"
IN_FILES=f"../IN/2026-00002-slurp/{SUB}/files"
OUT_DIR="../LEAK_DB/"
OUT_DB=f"{OUT_DIR}/{CASE}.sqlite3"
TS_START=-1
TS_END=-1

# Import data

In [ ]:
!ls $OUT_DIR


In [ ]:
#!rm $OUT_DIR/*

In [ ]:
!rm log.txt
!rm error_log.txt

# Open DB (or create it)

In [ ]:
if not os.path.exists(OUT_DB) :
    create_leak_db(OUT_DB)
# Open database
conn = sqlite3.connect(OUT_DB)
cur = conn.cursor()

# Import records

In [ ]:
files=sorted(glob(f"{IN_DIR}/*.jsonl"))
ffiles=sorted(glob(f"{IN_FILES}/*.jsonl"))


In [ ]:
ffiles

In [ ]:
#cur.execute("DELETE FROM 'entity'")


### Determine timestamps

In [ ]:
for file in ffiles :
    print(f"File: {file}") 

    records = []
    with open(file) as f:
        for line in f:
            record = json.loads(line)
            if TS_START < 0 or TS_START > record["_source"]["timestamp"] :
                TS_START = record["_source"]["timestamp"]
            if TS_END < 0 or TS_END < record["_source"]["timestamp"] :
                TS_END = record["_source"]["timestamp"]
            print(f"{TS_START} - {TS_END}", end="\r")
print(f"{TS_START} - {TS_END}")


In [ ]:
#record

### Import data

In [ ]:
count=0
skipped=0
for file in files :
    print(f"File: {file} - ", end="") 

    records = []
    with open(file) as f:
        for line in f:
            records.append(json.loads(line))
    print(f"{len(records)} records")
    with open("log.txt", "a") as lfile:
        lfile.write(f"File: {file} - {len(records)} records\n") 
    for row in records:
        login = row["_source"]["user"]
        if type(login) is str :
            user_ext = tldextract.extract(login)
            if user_ext.domain and user_ext.suffix :
                user_apex = f"{user_ext.domain}.{user_ext.suffix}"
            else:
                user_ext = None
                user_apex = None
        else:
            user_ext = None
            user_apex = None
        if user_apex:
            if re.search(r'%[0-9A-Fa-f]{2}|\+',user_apex) :
                user_apex = unquote(user_apex)
            user_apex = user_apex.strip()
        if type(row["_source"]["url"]) is str:
            url = row["_source"]["url"]
            url_ext = tldextract.extract(url)
            if url_ext.domain and url_ext.suffix:
                url_apex = f"{url_ext.domain}.{url_ext.suffix}"
            else:
                url_ext = None
                url_apex = None            
        else:
            url = None
            url_ext = None
            url_apex = None
        if url_apex :
            if re.search(r'%[0-9A-Fa-f]{2}|\+',url_apex) :
                url_apex = unquote(url_apex)
            url_apex = url_apex.strip()
        if type(login) is str:
            username = login
        else:
            username = None
        if type(row["_source"]["password"]) is str and row["_source"]["password"] != "":
            if PASSWORD_PREMASKED :
                passwd = None
                masked_passwd = row["_source"]["password"]
            else:
                passwd = row["_source"]["password"]
                masked_passwd = mask_password(passwd)
                row["_source"]["password"] = masked_passwd
        else:
            passwd = None
            masked_passwd = None
            row["_source"]["password"] = ""
        if username and passwd and is_email(username) and not PASSWORD_PREMASKED :
            nml_hash_str = nml_hash(username, passwd)
        else:
            nml_hash_str = None
        extradata = { 
            "source"                 : row["_source"].copy(),
            "harvested_on_or_after"  : datetime.fromtimestamp(TS_START, tz=timezone.utc).isoformat(),
            "harvested_before_or_on" : datetime.fromtimestamp(TS_END  , tz=timezone.utc).isoformat()
        }
        cur.execute("""
            SELECT count(*) FROM 'entity' WHERE username = ? and url_apex = ? and nml_hash = ? COLLATE NOCASE
        """, ( username, url_apex, nml_hash_str ) )
        if cur.fetchone()[0] == 0 :
            try:
                cur.execute("""
                    INSERT into 'entity' ( 
                        username, masked_passwd, nml_hash, url, 
                        email_apex, url_apex, ts_found, ts_leaked, 
                        has_name, has_dob, has_addr, has_phone, has_cc, has_bankacc,
                        has_ssn, has_ip, extra_data
                    ) values (
                        ?, ?, ?, ?, 
                        ?, ?, ?, ?, 
                        ?, ?, ?, ?, ?, ?, 
                        ?, ?, ?
                    )
                """ , (
                    username, masked_passwd, nml_hash_str, url,
                    user_apex, url_apex, None, None,
                    False, False, False, False, False, False, 
                    False, True, json.dumps(extradata, indent=2)
                ) )
            except Exception as e:
                with open('error_log.txt', 'a') as efile:
                    efile.write(f"{datetime.now()} - Error: {e}\n")
                print(f"\n{datetime.now()} - Error: {e}")
        else:
            skipped = skipped + 1
        count=count+1
        if count % 1_000 == 0 :
            print(f"{count:,}", end="\r")
        if count % 10_000 == 0 :
            conn.commit()
    with open("log.txt", "a") as lfile:
        lfile.write(f"\n{count:,}") 
conn.commit()
print(f"\n{count:,} - ({skipped:,} skipped).")

In [ ]:
conn.commit()